# Notebook 04: Clinical Entity Extraction with scispaCy

**Author:** Anthony Amit Biswas

## What this notebook does

Applies scispaCy to extract biomedical mentions from the preprocessed cohort using validated, resumable shard-based processing.


## 1. Install the Required Clinical NLP Packages

scispaCy and its biomedical language model are installed separately.

The package versions are fixed to maintain compatibility between NumPy, spaCy, scispaCy and the `en_core_sci_sm` biomedical model.

After running the installation cell, the Colab runtime must be restarted before continuing.

In [ ]:
# Installing the Required Clinical NLP Environment


%pip install --quiet --no-cache-dir \
    "numpy==1.26.4" \
    "pandas==2.2.2" \
    "spacy==3.7.5" \
    "scispacy==0.6.2"

%pip install --quiet --no-cache-dir \
    "https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz"

print("-" * 75)
print("INSTALLATION COMPLETED")
print("-" * 75)
print("Restart the Colab runtime before running the next section.")

## 2. Import the Required Libraries

After restarting the runtime, all Python libraries must be imported again before they can be used.

In [ ]:
# Importing the Required Libraries

import platform
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import spacy
import scispacy


print("-" * 75)
print("LIBRARIES IMPORTED SUCCESSFULLY")
print("-" * 75)

print(f"Python version    : {platform.python_version()}")
print(f"NumPy version     : {np.__version__}")
print(f"pandas version    : {pd.__version__}")
print(f"spaCy version     : {spacy.__version__}")
print(f"scispaCy version  : {scispacy.__version__}")

## 3. Load and Verify the scispaCy Biomedical Model

The `en_core_sci_sm` model is loaded after the required libraries have been imported.

The model provides tokenisation, part-of-speech tagging, lemmatisation, dependency parsing and biomedical mention detection.

In [ ]:
# Loading and Verifying the scispaCy Biomedical Language Model

nlp = spacy.load("en_core_sci_sm")

print("=" * 75)
print("SCISPACY MODEL VERIFICATION")
print("=" * 75)

print(f"Model name       : {nlp.meta.get('name')}")
print(f"Model version    : {nlp.meta.get('version')}")
print(f"Language         : {nlp.meta.get('lang')}")
print(f"Pipeline         : {nlp.pipe_names}")
print(f"Maximum length   : {nlp.max_length:,} characters")

## 4. Connect Google Drive

The dissertation dataset and all generated outputs are stored in the project directory on Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 5. Define Project File Paths

The project directory and the preprocessed discharge-summary dataset are defined using `pathlib`.

In [ ]:

PROJECT_DIRECTORY = Path(
    "/content/drive/MyDrive/Dissertation"
)

OUTPUT_DIRECTORY = (
    PROJECT_DIRECTORY / "outputs"
)

INPUT_FILE = (
    OUTPUT_DIRECTORY
    / "icu_discharge_summaries_preprocessed.csv.gz"
)


print("-" * 75)
print("PROJECT PATHS")
print("-" * 75)

print(f"Project directory : {PROJECT_DIRECTORY}")
print(f"Output directory  : {OUTPUT_DIRECTORY}")
print(f"Input file        : {INPUT_FILE}")
print(f"Input file exists : {INPUT_FILE.exists()}")

## 6. Load the Preprocessed ICU Discharge Summaries

The preprocessed cohort generated in Notebook 03 is loaded from a compressed CSV file.

In [ ]:
# Loading the Preprocessed Dataset

preprocessed_notes = pd.read_csv(
    INPUT_FILE,
    compression="gzip"
)


print("=" * 75)
print("DATASET LOADED")
print("=" * 75)

print(f"Rows    : {len(preprocessed_notes):,}")
print(f"Columns : {preprocessed_notes.shape[1]:,}")
print("\nColumn names:")
print(preprocessed_notes.columns.tolist())

## 7. Validate the Preprocessed Dataset

Validation checks are performed before applying the NLP pipeline.

The checks confirm that:

- all required columns are present;
- no cleaned notes are missing;
- no cleaned notes are empty;
- note identifiers are unique;
- the expected number of records is available.

In [ ]:
# Validating the Preprocessed Dataset

required_columns = {
    "note_id",
    "subject_id",
    "hadm_id",
    "text",
    "clean_text"
}

missing_columns = (
    required_columns
    - set(preprocessed_notes.columns)
)

missing_clean_notes = (
    preprocessed_notes["clean_text"]
    .isna()
    .sum()
)

empty_clean_notes = (
    preprocessed_notes["clean_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

duplicate_note_ids = (
    preprocessed_notes["note_id"]
    .duplicated()
    .sum()
)


print("-" * 75)
print("DATASET VALIDATION RESULTS")
print("-" * 75)

print(f"Rows                  : {len(preprocessed_notes):,}")
print(f"Missing columns       : {sorted(missing_columns)}")
print(f"Missing cleaned notes : {missing_clean_notes:,}")
print(f"Empty cleaned notes   : {empty_clean_notes:,}")
print(f"Duplicate note IDs    : {duplicate_note_ids:,}")

**Assertions**

In [ ]:
# Applying Strict Validation Assertions


EXPECTED_ROW_COUNT = 65_323

assert not missing_columns, (
    f"Missing required columns: {sorted(missing_columns)}"
)

assert len(preprocessed_notes) == EXPECTED_ROW_COUNT, (
    f"Expected {EXPECTED_ROW_COUNT:,} rows, "
    f"but found {len(preprocessed_notes):,}."
)

assert missing_clean_notes == 0, (
    "Missing values were found in clean_text."
)

assert empty_clean_notes == 0, (
    "Empty cleaned notes were found."
)

assert duplicate_note_ids == 0, (
    "Duplicate note IDs were found."
)

print("All dataset validation checks passed successfully.")

## 8. Inspect the Length of the Preprocessed Notes

Clinical discharge summaries vary substantially in length.

Character and word counts are calculated to identify unusually short or long documents and to confirm compatibility with the scispaCy input-length limit.

In [ ]:
# Calculating Character and Word Counts

preprocessed_notes["clean_character_count"] = (
    preprocessed_notes["clean_text"]
    .astype(str)
    .str.len()
)

preprocessed_notes["clean_word_count"] = (
    preprocessed_notes["clean_text"]
    .astype(str)
    .str.split()
    .str.len()
)


print("-" * 75)
print("NOTE-LENGTH SUMMARY")
print("-" * 75)

display(
    preprocessed_notes[
        [
            "clean_character_count",
            "clean_word_count"
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

**Model-length checking**

In [ ]:
# Comparing Note Lengths with the spaCy Maximum Input Length

maximum_note_length = (
    preprocessed_notes["clean_character_count"]
    .max()
)

longest_note_index = (
    preprocessed_notes["clean_character_count"]
    .idxmax()
)

notes_exceeding_model_limit = (
    preprocessed_notes["clean_character_count"]
    .gt(nlp.max_length)
    .sum()
)


print("-" * 75)
print("MODEL-LENGTH COMPATIBILITY CHECK")
print("-" * 75)

print(
    f"spaCy maximum length      : "
    f"{nlp.max_length:,} characters"
)

print(
    f"Longest note length       : "
    f"{maximum_note_length:,} characters"
)

print(
    f"Longest note row index    : "
    f"{longest_note_index}"
)

print(
    f"Notes exceeding the limit : "
    f"{notes_exceeding_model_limit:,}"
)

## 9. Select a Representative ICU Discharge Summary

Before applying scispaCy to the complete dataset, the pipeline is tested on one representative discharge summary.

A note with a character length close to the median is selected. This avoids using an unusually short or unusually long document and provides a realistic example for initial testing.

In [ ]:
# Selecting a Representative Note Close to the Median Length

median_character_count = (
    preprocessed_notes["clean_character_count"]
    .median()
)

representative_note_index = (
    preprocessed_notes["clean_character_count"]
    .sub(median_character_count)
    .abs()
    .idxmin()
)

representative_note = (
    preprocessed_notes
    .loc[representative_note_index]
    .copy()
)

representative_text = str(
    representative_note["clean_text"]
)


print("-" * 75)
print("REPRESENTATIVE NOTE SELECTED")
print("-" * 75)

print(f"Row index       : {representative_note_index}")
print(f"Note ID         : {representative_note['note_id']}")
print(f"Subject ID      : {representative_note['subject_id']}")
print(f"Admission ID    : {representative_note['hadm_id']}")
print(
    f"Character count : "
    f"{representative_note['clean_character_count']:,}"
)
print(
    f"Word count      : "
    f"{representative_note['clean_word_count']:,}"
)

## 10. Preview the Representative Note

Only the beginning of the selected discharge summary is displayed to verify that the text was loaded correctly. The complete note is not printed because discharge summaries can be lengthy.

In [ ]:
# Previewing the Beginning of the Representative Note

PREVIEW_LENGTH = 1_500

print("-" * 75)
print("REPRESENTATIVE NOTE PREVIEW")
print("-" * 75)

print(representative_text[:PREVIEW_LENGTH])

if len(representative_text) > PREVIEW_LENGTH:
    print("\n[Preview truncated]")

## 11. Process One ICU Discharge Summary with scispaCy

The representative discharge summary is processed using the `en_core_sci_sm` biomedical NLP model.

The processing time, token count, sentence count and number of detected biomedical mentions are recorded to confirm that the pipeline operates correctly before batch processing.

In [ ]:
# Processing One Representative ICU Discharge Summary

processing_start_time = time.perf_counter()

representative_document = nlp(
    representative_text
)

processing_end_time = time.perf_counter()

processing_duration = (
    processing_end_time
    - processing_start_time
)

detected_entities = list(
    representative_document.ents
)

sentence_count = sum(
    1 for _ in representative_document.sents
)


print("-" * 75)
print("REPRESENTATIVE NOTE PROCESSING RESULTS")
print("-" * 75)

print(f"Processing time     : {processing_duration:.2f} seconds")
print(f"Character count     : {len(representative_text):,}")
print(f"Token count         : {len(representative_document):,}")
print(f"Sentence count      : {sentence_count:,}")
print(f"Detected mentions   : {len(detected_entities):,}")

**Displaying a sample of detected mentions**

In [ ]:
# Displaying a Sample of Detected Biomedical Mentions

MAXIMUM_ENTITIES_TO_DISPLAY = 50

print("-" * 75)
print("SAMPLE OF DETECTED BIOMEDICAL MENTIONS")
print("-" * 75)

if not detected_entities:
    print("No biomedical mentions were detected.")

else:
    for entity_number, entity in enumerate(
        detected_entities[:MAXIMUM_ENTITIES_TO_DISPLAY],
        start=1
    ):
        print(
            f"{entity_number:>3}. "
            f"{entity.text.strip():<45} "
            f"Start: {entity.start_char:<6} "
            f"End: {entity.end_char:<6} "
            f"Label: {entity.label_}"
        )

    remaining_entities = (
        len(detected_entities)
        - MAXIMUM_ENTITIES_TO_DISPLAY
    )

    if remaining_entities > 0:
        print(
            f"\n{remaining_entities:,} additional mentions "
            "were not displayed."
        )

## 12. Structure the Extracted Biomedical Mentions

The biomedical mentions detected in the representative discharge summary are converted into a structured table.

For each mention, the table records:

- the note identifier;
- the original entity text;
- a normalised lowercase form;
- the character offsets;
- the entity label.

The normalised form will later support frequency analysis and duplicate handling.

In [ ]:
# Converting Representative-Note Entities into a Structured DataFrame

representative_entity_records = []

for entity in representative_document.ents:

    entity_text = entity.text.strip()

    if not entity_text:
        continue

    representative_entity_records.append(
        {
            "note_id": representative_note["note_id"],
            "subject_id": representative_note["subject_id"],
            "hadm_id": representative_note["hadm_id"],
            "entity_text": entity_text,
            "normalized_entity": entity_text.lower(),
            "start_char": entity.start_char,
            "end_char": entity.end_char,
            "entity_label": entity.label_
        }
    )


representative_entities_df = pd.DataFrame(
    representative_entity_records
)


print("-" * 75)
print("STRUCTURED REPRESENTATIVE-NOTE ENTITIES")
print("-" * 75)

print(f"Entity rows       : {len(representative_entities_df):,}")
print(f"Entity columns    : {representative_entities_df.shape[1]:,}")
print(f"Unique mentions   : "
      f"{representative_entities_df['normalized_entity'].nunique():,}")

display(
    representative_entities_df.head(20)
)

## 13. Examine Repeated Biomedical Mentions

A discharge summary can contain the same biomedical concept multiple times.

The frequency of normalised mentions is examined to understand repetition within the representative note. These repeated occurrences are retained because their character offsets and textual contexts differ.

In [ ]:
# Testing the Most Frequent Normalised Mentions

representative_entity_frequencies = (
    representative_entities_df["normalized_entity"]
    .value_counts()
    .rename_axis("normalized_entity")
    .reset_index(name="frequency")
)

print("-" * 75)
print("MOST FREQUENT MENTIONS IN THE REPRESENTATIVE NOTE")
print("-" * 75)

display(
    representative_entity_frequencies.head(25)
)

## 14. Create a Small Batch for Pipeline Benchmarking

Before processing the complete cohort, a reproducible sample of discharge summaries is selected.

The sample is used to:

- verify that multiple notes can be processed successfully;
- estimate processing speed;
- calculate the number of mentions extracted per note;
- identify possible failures before full-cohort processing.

In [ ]:
# Selecting a Reproducible Benchmark Sample

BENCHMARK_SAMPLE_SIZE = 100
RANDOM_SEED = 42

benchmark_notes = (
    preprocessed_notes
    .sample(
        n=BENCHMARK_SAMPLE_SIZE,
        random_state=RANDOM_SEED
    )
    .reset_index(drop=True)
    .copy()
)


print("-" * 75)
print("BENCHMARK SAMPLE")
print("-" * 75)

print(f"Sample size             : {len(benchmark_notes):,}")
print(
    f"Minimum note length     : "
    f"{benchmark_notes['clean_character_count'].min():,} characters"
)
print(
    f"Median note length      : "
    f"{benchmark_notes['clean_character_count'].median():,.0f} characters"
)
print(
    f"Maximum note length     : "
    f"{benchmark_notes['clean_character_count'].max():,} characters"
)

## 15. Benchmark scispaCy Batch Processing

The benchmark notes are processed using `nlp.pipe`, spaCy's batch-processing interface.

Only the components required for biomedical mention detection are retained during this benchmark. Part-of-speech tagging, lemmatisation and dependency parsing are disabled because the immediate task is entity extraction rather than full linguistic analysis.

The named-entity recognition component and its shared `tok2vec` representation remain active.

In [ ]:
# Benchmark Batch Biomedical Mention Extraction

BATCH_SIZE = 8

benchmark_texts = (
    benchmark_notes["clean_text"]
    .astype(str)
    .tolist()
)

benchmark_results = []

benchmark_start_time = time.perf_counter()

with nlp.select_pipes(
    disable=[
        "tagger",
        "attribute_ruler",
        "lemmatizer",
        "parser"
    ]
):

    for row_position, document in enumerate(
        nlp.pipe(
            benchmark_texts,
            batch_size=BATCH_SIZE
        )
    ):

        note_row = benchmark_notes.iloc[row_position]

        benchmark_results.append(
            {
                "note_id": note_row["note_id"],
                "subject_id": note_row["subject_id"],
                "hadm_id": note_row["hadm_id"],
                "character_count": len(note_row["clean_text"]),
                "token_count": len(document),
                "entity_count": len(document.ents)
            }
        )

benchmark_end_time = time.perf_counter()

benchmark_duration = (
    benchmark_end_time
    - benchmark_start_time
)

benchmark_results_df = pd.DataFrame(
    benchmark_results
)


print("-" * 75)
print("BENCHMARK PROCESSING RESULTS")
print("-" * 75)

print(f"Notes processed        : {len(benchmark_results_df):,}")
print(f"Total processing time  : {benchmark_duration:.2f} seconds")
print(
    f"Average time per note  : "
    f"{benchmark_duration / len(benchmark_results_df):.3f} seconds"
)
print(
    f"Processing rate        : "
    f"{len(benchmark_results_df) / benchmark_duration:.2f} notes/second"
)
print(
    f"Total detected mentions: "
    f"{benchmark_results_df['entity_count'].sum():,}"
)
print(
    f"Mean mentions per note : "
    f"{benchmark_results_df['entity_count'].mean():.2f}"
)
print(
    f"Median mentions/note   : "
    f"{benchmark_results_df['entity_count'].median():.2f}"
)

**Validating the benchmark results**

In [ ]:
# Validating Benchmark Processing Results

assert len(benchmark_results_df) == BENCHMARK_SAMPLE_SIZE, (
    "The number of processed notes does not match the benchmark sample size."
)

assert benchmark_results_df["note_id"].isna().sum() == 0, (
    "Missing note identifiers were found."
)

assert benchmark_results_df["note_id"].duplicated().sum() == 0, (
    "Duplicate note identifiers were found."
)

assert benchmark_results_df["entity_count"].isna().sum() == 0, (
    "Missing entity counts were found."
)

assert (benchmark_results_df["entity_count"] >= 0).all(), (
    "Invalid negative entity counts were found."
)

print("All benchmark processing checks passed successfully.")

**Inspecting benchmark statistics**

In [ ]:
# Summarising Benchmark Extraction Statistics

print("-" * 75)
print("BENCHMARK EXTRACTION SUMMARY")
print("-" * 75)

display(
    benchmark_results_df[
        [
            "character_count",
            "token_count",
            "entity_count"
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
    .round(2)
)

## 16. Batch Clinical Entity Extraction

The complete preprocessed ICU discharge-summary cohort is processed using the scispaCy biomedical language model.

Biomedical mentions extracted from each discharge summary are written directly to disk in batches rather than being retained in memory. This streaming approach reduces memory consumption and improves robustness when processing a large clinical corpus.

In [ ]:
# Importing Additional Libraries


import csv
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# Defining Output File

ENTITY_OUTPUT_FILE = (
    OUTPUT_DIRECTORY /
    "scispacy_raw_entities.csv"
)

print("Output file:")
print(ENTITY_OUTPUT_FILE)

In [ ]:
# Creating CSV Header

with open(
    ENTITY_OUTPUT_FILE,
    mode="w",
    newline="",
    encoding="utf-8"
) as csv_file:

    writer = csv.writer(csv_file)

    writer.writerow(
        [
            "note_id",
            "subject_id",
            "hadm_id",
            "entity_text",
            "normalized_entity",
            "start_char",
            "end_char",
            "entity_label"
        ]
    )

print("CSV file initialised successfully.")

In [ ]:
# Configuring Batch Processing

BATCH_SIZE = 8

processed_notes = 0
processed_entities = 0

print(f"Batch size: {BATCH_SIZE}")

In [ ]:
# Streaming Biomedical Mentions to CSV

processing_start_time = time.perf_counter()

texts = (
    preprocessed_notes["clean_text"]
    .astype(str)
    .tolist()
)

metadata = preprocessed_notes[
    [
        "note_id",
        "subject_id",
        "hadm_id"
    ]
].reset_index(drop=True)

## 16. Extract Biomedical Mentions from the Full Cohort

The complete ICU discharge-summary cohort is processed using the scispaCy
`en_core_sci_sm` model.

To avoid retaining millions of extracted mentions in memory, the notes are
processed in fixed-size shards. Each completed shard is saved as a compressed
CSV file.

This design provides several advantages:

- controlled memory consumption;
- reduced output-file size through compression;
- recovery after a Colab disconnection;
- automatic skipping of previously completed shards;
- preservation of note identifiers and character offsets.

**Imports and output directories**

In [ ]:
# Importing Libraries Required for Resumable Extraction

import csv
import gzip
import math
import os
import time
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm


# Defining Extraction Output Locations

ENTITY_SHARD_DIRECTORY = (
    OUTPUT_DIRECTORY / "scispacy_entity_shards"
)

MANIFEST_FILE = (
    OUTPUT_DIRECTORY / "scispacy_extraction_manifest.csv"
)

ENTITY_SHARD_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 75)
print("EXTRACTION OUTPUT LOCATIONS")
print("=" * 75)

print(f"Shard directory : {ENTITY_SHARD_DIRECTORY}")
print(f"Manifest file   : {MANIFEST_FILE}")

In [ ]:
# Configuring Full-Cohort Extraction

NOTES_PER_SHARD = 250
NLP_BATCH_SIZE = 8

TOTAL_NOTES = len(preprocessed_notes)

TOTAL_SHARDS = math.ceil(
    TOTAL_NOTES / NOTES_PER_SHARD
)


ENTITY_COLUMNS = [
    "note_id",
    "subject_id",
    "hadm_id",
    "entity_text",
    "normalized_entity",
    "start_char",
    "end_char",
    "entity_label"
]


print("-" * 75)
print("EXTRACTION CONFIGURATION")
print("-" * 75)

print(f"Total notes          : {TOTAL_NOTES:,}")
print(f"Notes per shard      : {NOTES_PER_SHARD:,}")
print(f"Total shards         : {TOTAL_SHARDS:,}")
print(f"spaCy batch size     : {NLP_BATCH_SIZE:,}")

**Creating the manifest if required**

In [ ]:
# Initialising the Extraction Manifest

MANIFEST_COLUMNS = [
    "shard_number",
    "start_row",
    "end_row_exclusive",
    "notes_processed",
    "entities_extracted",
    "processing_seconds",
    "notes_per_second",
    "output_file"
]


if not MANIFEST_FILE.exists():

    pd.DataFrame(
        columns=MANIFEST_COLUMNS
    ).to_csv(
        MANIFEST_FILE,
        index=False
    )

    print("A new extraction manifest was created.")

else:
    print("An existing extraction manifest was found.")
    print("Completed shards will be skipped automatically.")

**Defining a function for one shard**

In [ ]:
# Defining the Shard Extraction Function

def extract_entity_shard(
    shard_notes: pd.DataFrame,
    shard_number: int,
    output_file: Path,
    batch_size: int = 8
) -> dict:
    """
    Process one shard of ICU discharge summaries and save all detected
    biomedical mentions to a compressed CSV file.

    Parameters
    ----------
    shard_notes:
        DataFrame containing note identifiers and clean clinical text.

    shard_number:
        Sequential number identifying the current shard.

    output_file:
        Final compressed CSV path for the extracted mentions.

    batch_size:
        Number of documents supplied to spaCy at one time.

    Returns
    -------
    dict
        Summary statistics describing the completed shard.
    """

    shard_start_time = time.perf_counter()

    entity_records = []

    # Attaching the note metadata to each text so that the metadata remains aligned
    # with the corresponding spaCy document during batch processing.
    text_and_context = [
        (
            str(row.clean_text),
            {
                "note_id": row.note_id,
                "subject_id": row.subject_id,
                "hadm_id": row.hadm_id
            }
        )
        for row in shard_notes.itertuples(index=False)
    ]

    # Only tok2vec and NER are required here for biomedical mention extraction.
    with nlp.select_pipes(
        disable=[
            "tagger",
            "attribute_ruler",
            "lemmatizer",
            "parser"
        ]
    ):

        processed_documents = nlp.pipe(
            text_and_context,
            as_tuples=True,
            batch_size=batch_size
        )

        for document, context in processed_documents:

            for entity in document.ents:

                entity_text = entity.text.strip()

                if not entity_text:
                    continue

                entity_records.append(
                    {
                        "note_id": context["note_id"],
                        "subject_id": context["subject_id"],
                        "hadm_id": context["hadm_id"],
                        "entity_text": entity_text,
                        "normalized_entity": entity_text.casefold(),
                        "start_char": entity.start_char,
                        "end_char": entity.end_char,
                        "entity_label": entity.label_
                    }
                )

    entity_dataframe = pd.DataFrame(
        entity_records,
        columns=ENTITY_COLUMNS
    )

    # Writing to a temporary file first. The temporary file is renamed only
    # after the complete shard has been written successfully.
    temporary_file = Path(
        str(output_file) + ".tmp"
    )

    with gzip.open(
        temporary_file,
        mode="wt",
        encoding="utf-8",
        newline=""
    ) as compressed_file:

        entity_dataframe.to_csv(
            compressed_file,
            index=False,
            quoting=csv.QUOTE_MINIMAL
        )

    os.replace(
        temporary_file,
        output_file
    )

    shard_duration = (
        time.perf_counter()
        - shard_start_time
    )

    notes_processed = len(shard_notes)

    return {
        "shard_number": shard_number,
        "notes_processed": notes_processed,
        "entities_extracted": len(entity_dataframe),
        "processing_seconds": shard_duration,
        "notes_per_second": (
            notes_processed / shard_duration
            if shard_duration > 0
            else 0
        ),
        "output_file": output_file.name
    }

**Small extraction test**

In [ ]:
# Testing the Shard Extraction Function on Ten Notes

TEST_OUTPUT_FILE = (
    ENTITY_SHARD_DIRECTORY /
    "test_shard_10_notes.csv.gz"
)

test_notes = (
    preprocessed_notes
    .iloc[:10]
    .copy()
)

test_result = extract_entity_shard(
    shard_notes=test_notes,
    shard_number=0,
    output_file=TEST_OUTPUT_FILE,
    batch_size=NLP_BATCH_SIZE
)


print("-" * 75)
print("TEST SHARD COMPLETED")
print("-" * 75)

for key, value in test_result.items():
    print(f"{key:<22}: {value}")

print(f"\nOutput exists         : {TEST_OUTPUT_FILE.exists()}")
print(
    f"Compressed file size  : "
    f"{TEST_OUTPUT_FILE.stat().st_size / 1_000_000:.2f} MB"
)

**Reading and validating the test output**

In [ ]:
# Validating the Ten-Note Test Output

test_entities = pd.read_csv(
    TEST_OUTPUT_FILE,
    compression="gzip"
)


print("-" * 75)
print("TEST OUTPUT VALIDATION")
print("-" * 75)

print(f"Extracted rows        : {len(test_entities):,}")
print(f"Extracted columns     : {test_entities.shape[1]:,}")
print(
    f"Unique note IDs       : "
    f"{test_entities['note_id'].nunique():,}"
)
print(
    f"Missing entity text   : "
    f"{test_entities['entity_text'].isna().sum():,}"
)
print(
    f"Invalid offsets       : "
    f"{(test_entities['end_char'] <= test_entities['start_char']).sum():,}"
)

display(
    test_entities.head(20)
)

**Strict test assertions**

In [ ]:
# Applying Strict Test Assertions

assert list(test_entities.columns) == ENTITY_COLUMNS, (
    "The extracted entity columns are incorrect."
)

assert test_entities["note_id"].nunique() == len(test_notes), (
    "One or more test notes produced no recorded output."
)

assert test_entities["entity_text"].isna().sum() == 0, (
    "Missing entity text was found."
)

assert test_entities["normalized_entity"].isna().sum() == 0, (
    "Missing normalised entity text was found."
)

assert (
    test_entities["end_char"]
    > test_entities["start_char"]
).all(), (
    "Invalid entity character offsets were found."
)

print("All test-shard validation checks passed successfully.")

## 17. Run Resumable Full-Cohort Extraction

Each shard is processed and saved independently. A shard is skipped when its
completed output file already exists.

Consequently, if the Colab runtime disconnects, the extraction can be resumed
by rerunning this cell. Previously completed shards will not be processed again.

**Full extraction loop**

In [ ]:
# Running Resumable Full-Cohort Biomedical Mention Extraction
RUN_FULL_EXTRACTION = False

full_extraction_start_time = time.perf_counter()

completed_during_this_run = 0
skipped_existing_shards = 0
entities_during_this_run = 0


for shard_number in tqdm(
    range(TOTAL_SHARDS),
    desc="Processing entity shards"
):

    start_row = (
        shard_number
        * NOTES_PER_SHARD
    )

    end_row = min(
        start_row + NOTES_PER_SHARD,
        TOTAL_NOTES
    )

    shard_output_file = (
        ENTITY_SHARD_DIRECTORY
        / f"scispacy_entities_shard_{shard_number:04d}.csv.gz"
    )

    # A successfully completed shard is not processed again.
    if shard_output_file.exists():

        skipped_existing_shards += 1
        continue

    shard_notes = (
        preprocessed_notes
        .iloc[start_row:end_row]
        [
            [
                "note_id",
                "subject_id",
                "hadm_id",
                "clean_text"
            ]
        ]
        .copy()
    )

    shard_result = extract_entity_shard(
        shard_notes=shard_notes,
        shard_number=shard_number,
        output_file=shard_output_file,
        batch_size=NLP_BATCH_SIZE
    )

    shard_result.update(
        {
            "start_row": start_row,
            "end_row_exclusive": end_row
        }
    )

    manifest_row = pd.DataFrame(
        [
            {
                column: shard_result[column]
                for column in MANIFEST_COLUMNS
            }
        ]
    )

    manifest_row.to_csv(
        MANIFEST_FILE,
        mode="a",
        header=False,
        index=False
    )

    completed_during_this_run += 1

    entities_during_this_run += (
        shard_result["entities_extracted"]
    )


full_extraction_duration = (
    time.perf_counter()
    - full_extraction_start_time
)

print("FULL EXTRACTION RUN SUMMARY")


print(
    f"Shards completed this run : "
    f"{completed_during_this_run:,}"
)

print(
    f"Existing shards skipped   : "
    f"{skipped_existing_shards:,}"
)

print(
    f"Entities extracted now     : "
    f"{entities_during_this_run:,}"
)

print(
    f"Run duration               : "
    f"{full_extraction_duration / 60:.2f} minutes"
)

**Checking if the extraction and the files are okay after runtime got disconnected!!!**

In [ ]:
import pandas as pd

manifest = pd.read_csv(MANIFEST_FILE)

print("Manifest rows:", len(manifest))
print("Total notes:", manifest["notes_processed"].sum())
print("Total entities:", manifest["entities_extracted"].sum())

In [ ]:
from pathlib import Path

total_size = sum(
    f.stat().st_size
    for f in ENTITY_SHARD_DIRECTORY.glob("*.csv.gz")
)

print(f"Shard files: {len(list(ENTITY_SHARD_DIRECTORY.glob('*.csv.gz')))}")
print(f"Total size: {total_size / (1024**3):.2f} GB")

In [ ]:
from pathlib import Path

for file in sorted(ENTITY_SHARD_DIRECTORY.glob("*.csv.gz")):
    print(file.name)

In [ ]:
real_shards = list(
    ENTITY_SHARD_DIRECTORY.glob(
        "scispacy_entities_shard_*.csv.gz"
    )
)

print("Real shard files:", len(real_shards))

## 18. Validate the Completed scispaCy Extraction

The completed extraction is validated before any statistical analysis is performed.

The validation checks confirm that:

- all expected shard files are present;
- shard numbering is complete;
- no unexpected shard files are included;
- the extraction manifest contains one record per completed shard;
- the manifest totals agree with the full extraction output.

**Locating all completed shard files**

In [ ]:
import re
from pathlib import Path

entity_shard_files = sorted(
    ENTITY_SHARD_DIRECTORY.glob(
        "scispacy_entities_shard_*.csv.gz"
    )
)


print("-" * 75)
print("ENTITY SHARD FILE CHECK")
print("-" * 75)

print(f"Expected shard files : {TOTAL_SHARDS:,}")
print(f"Located shard files  : {len(entity_shard_files):,}")

## 19. Validation of the Extraction Manifest

The extraction manifest records the processing outcome for every shard.  
This section verifies that:

- the manifest contains the expected number of shards;
- each shard number appears exactly once;
- no shard records are duplicated;
- the recorded note ranges cover the complete cohort;
- the total number of processed notes matches the source dataset;
- the total number of extracted biomedical mentions is internally consistent.

In [ ]:
# Validate the Extraction Manifest

manifest = pd.read_csv(MANIFEST_FILE)

print("-" * 75)
print("EXTRACTION MANIFEST CHECK")
print("-" * 75)

print(f"Manifest rows        : {len(manifest):,}")
print(f"Manifest columns     : {len(manifest.columns):,}")
print(f"Expected shard rows  : {TOTAL_SHARDS:,}")

print("\nManifest columns:")
for column in manifest.columns:
    print(f"  - {column}")

In [ ]:
# Validating Manifest Completeness and Shard Numbering

required_manifest_columns = {
    "shard_number",
    "start_row",
    "end_row_exclusive",
    "notes_processed",
    "entities_extracted"
}

missing_manifest_columns = (
    required_manifest_columns
    - set(manifest.columns)
)

assert not missing_manifest_columns, (
    "The manifest is missing required columns: "
    f"{sorted(missing_manifest_columns)}"
)


# Basic manifest checking

assert len(manifest) == TOTAL_SHARDS, (
    f"Expected {TOTAL_SHARDS} manifest rows, "
    f"but found {len(manifest)}."
)

assert manifest["shard_number"].is_unique, (
    "Duplicate shard numbers were found in the manifest."
)

assert not manifest[list(required_manifest_columns)].isna().any().any(), (
    "Missing values were found in required manifest columns."
)

# Validating expected shard-number sequence

expected_shard_numbers = set(range(TOTAL_SHARDS))

recorded_shard_numbers = set(
    manifest["shard_number"].astype(int)
)

missing_manifest_shards = sorted(
    expected_shard_numbers
    - recorded_shard_numbers
)

unexpected_manifest_shards = sorted(
    recorded_shard_numbers
    - expected_shard_numbers
)

assert not missing_manifest_shards, (
    f"Missing manifest shard numbers: "
    f"{missing_manifest_shards}"
)

assert not unexpected_manifest_shards, (
    f"Unexpected manifest shard numbers: "
    f"{unexpected_manifest_shards}"
)

# Sorting manifest into processing order


manifest = (
    manifest
    .sort_values("shard_number")
    .reset_index(drop=True)
)

print("-" * 75)
print("MANIFEST STRUCTURE VALIDATION")
print("-" * 75)

print(f"Expected manifest rows : {TOTAL_SHARDS:,}")
print(f"Located manifest rows  : {len(manifest):,}")
print(f"Unique shard numbers   : {manifest['shard_number'].nunique():,}")
print(f"Missing shard numbers  : {len(missing_manifest_shards):,}")
print(f"Unexpected shards      : {len(unexpected_manifest_shards):,}")

print("\nManifest structure validation passed.")

In [ ]:
# Validating Note Coverage and Extraction Totals

manifest_total_notes = int(
    manifest["notes_processed"].sum()
)

manifest_total_entities = int(
    manifest["entities_extracted"].sum()
)

manifest_first_row = int(
    manifest["start_row"].min()
)

manifest_final_row = int(
    manifest["end_row_exclusive"].max()
)

# Each manifest row should describe a valid range.
assert (
    manifest["end_row_exclusive"]
    > manifest["start_row"]
).all(), (
    "One or more manifest rows contain an invalid note range."
)

# The recorded range length should equal notes processed.
recorded_range_lengths = (
    manifest["end_row_exclusive"]
    - manifest["start_row"]
)

assert (
    recorded_range_lengths
    == manifest["notes_processed"]
).all(), (
    "One or more manifest note ranges do not match notes_processed."
)

# Consecutive shards should connect without gaps or overlaps.
previous_end_rows = (
    manifest["end_row_exclusive"]
    .iloc[:-1]
    .to_numpy()
)

next_start_rows = (
    manifest["start_row"]
    .iloc[1:]
    .to_numpy()
)

assert (
    previous_end_rows
    == next_start_rows
).all(), (
    "A gap or overlap was detected between consecutive shards."
)

assert manifest_first_row == 0, (
    f"The first shard begins at row {manifest_first_row}, "
    "rather than row 0."
)

assert manifest_final_row == TOTAL_NOTES, (
    f"The manifest ends at row {manifest_final_row}, "
    f"but the cohort contains {TOTAL_NOTES} notes."
)

assert manifest_total_notes == TOTAL_NOTES, (
    f"Manifest total notes: {manifest_total_notes}; "
    f"expected: {TOTAL_NOTES}."
)

assert manifest_total_entities > 0, (
    "The manifest reports zero extracted entities."
)

print("-" * 75)
print("MANIFEST COHORT COVERAGE CHECK")
print("-" * 75)

print(f"First source row          : {manifest_first_row:,}")
print(f"Final exclusive row       : {manifest_final_row:,}")
print(f"Total notes in manifest   : {manifest_total_notes:,}")
print(f"Expected cohort notes     : {TOTAL_NOTES:,}")
print(f"Total extracted mentions  : {manifest_total_entities:,}")
print(f"Range gaps or overlaps    : 0")

print("\nManifest cohort coverage validation passed.")

## 20. Processing Summary Statistics

This section summarizes the computational characteristics of the extraction
pipeline, including the number of processed shards, notes, biomedical mentions,
and descriptive statistics of the extracted mentions per shard.

In [ ]:
# Processing Summary Statistics

processing_statistics = {
    "Total shards": len(manifest),
    "Total notes": int(manifest["notes_processed"].sum()),
    "Total biomedical mentions": int(manifest["entities_extracted"].sum()),
    "Mean mentions per shard": manifest["entities_extracted"].mean(),
    "Median mentions per shard": manifest["entities_extracted"].median(),
    "Minimum mentions per shard": manifest["entities_extracted"].min(),
    "Maximum mentions per shard": manifest["entities_extracted"].max(),
    "Standard deviation": manifest["entities_extracted"].std(),
    "Mean mentions per note": (
        manifest["entities_extracted"].sum()
        / manifest["notes_processed"].sum()
    )
}

print("-" * 75)
print("PROCESSING SUMMARY")
print("-" * 75)

for key, value in processing_statistics.items():

    if isinstance(value, float):
        print(f"{key:<32}: {value:,.2f}")
    else:
        print(f"{key:<32}: {value:,}")

## 21. Streaming Biomedical Mention Frequency Analysis

A memory-efficient frequency analysis was performed over the extracted
biomedical mentions. Each production shard was loaded separately, mention
strings were normalised by removing surrounding whitespace, consolidating
internal whitespace, and converting text to lowercase.

Because `en_core_sci_sm` produces generic `ENTITY` spans, the resulting table
represents frequently extracted biomedical mentions rather than validated
diagnoses, medications, procedures, or clinical complications.

**Locating the production shards and identify the mention column**

In [ ]:
# Locate Production Entity Shards

from collections import Counter
from pathlib import Path
import re

production_shard_files = sorted(
    ENTITY_SHARD_DIRECTORY.glob(
        "scispacy_entities_shard_*.csv.gz"
    )
)

assert len(production_shard_files) == TOTAL_SHARDS, (
    f"Expected {TOTAL_SHARDS} production shards, "
    f"but found {len(production_shard_files)}."
)

# Inspect the first production shard.
sample_shard = pd.read_csv(
    production_shard_files[0],
    nrows=5
)

print("-" * 75)
print("ENTITY SHARD SCHEMA")
print("-" * 75)

print(f"Production shards located : {len(production_shard_files):,}")
print("\nAvailable columns:")

for column in sample_shard.columns:
    print(f"  - {column}")

**Automatically identifying the biomedical mention column**

In [ ]:
# Identify Biomedical Mention Text Column

candidate_mention_columns = [
    "entity_text",
    "entity",
    "mention",
    "mention_text",
    "text"
]

ENTITY_TEXT_COLUMN = next(
    (
        column
        for column in candidate_mention_columns
        if column in sample_shard.columns
    ),
    None
)

assert ENTITY_TEXT_COLUMN is not None, (
    "The biomedical mention text column could not be identified. "
    f"Available columns: {list(sample_shard.columns)}"
)

print(f"Biomedical mention column: {ENTITY_TEXT_COLUMN}")

**Running the streaming frequency analysis**

In [ ]:
# Streaming Biomedical Mention Frequency Analysis
RUN_FULL_EXTRACTION = False

mention_counter = Counter()

rows_scanned = 0
blank_mentions_removed = 0

for shard_file in tqdm(
    production_shard_files,
    desc="Analysing mention frequencies"
):

    shard_mentions = pd.read_csv(
        shard_file,
        usecols=[ENTITY_TEXT_COLUMN],
        dtype={ENTITY_TEXT_COLUMN: "string"}
    )

    rows_scanned += len(shard_mentions)

    # Removing missing values.
    normalised_mentions = (
        shard_mentions[ENTITY_TEXT_COLUMN]
        .dropna()
        .astype(str)
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .str.lower()
    )

    # Removing blank strings created by normalisation.
    blank_mask = normalised_mentions.eq("")

    blank_mentions_removed += int(
        blank_mask.sum()
    )

    normalised_mentions = normalised_mentions[
        ~blank_mask
    ]

    # Aggregating within the current shard before updating
    # the global frequency counter.
    shard_frequencies = (
        normalised_mentions
        .value_counts()
    )

    mention_counter.update(
        shard_frequencies.to_dict()
    )

print("-" * 75)
print("STREAMING FREQUENCY ANALYSIS COMPLETE")
print("-" * 75)

print(f"Shard files processed      : {len(production_shard_files):,}")
print(f"Entity rows scanned        : {rows_scanned:,}")
print(f"Blank mentions removed     : {blank_mentions_removed:,}")
print(f"Valid mentions counted     : {sum(mention_counter.values()):,}")
print(f"Unique normalised mentions : {len(mention_counter):,}")

### Accounting for Missing Biomedical Mention Values

The streaming analysis removed missing values before normalising and counting
biomedical mentions. This section calculates the number of missing entity-text
values and confirms that every extracted entity row is fully accounted for.

In [ ]:
# Accounting for Missing Biomedical Mention Values

missing_mentions_removed = (
    rows_scanned
    - valid_mentions_counted
    - blank_mentions_removed
)

assert missing_mentions_removed >= 0, (
    "The calculated number of missing mentions cannot be negative."
)

accounted_entity_rows = (
    valid_mentions_counted
    + missing_mentions_removed
    + blank_mentions_removed
)

assert accounted_entity_rows == rows_scanned, (
    f"Only {accounted_entity_rows:,} of "
    f"{rows_scanned:,} entity rows were accounted for."
)

print("-" * 75)
print("FINAL MENTION ACCOUNTING")
print("-" * 75)

print(f"Entity rows scanned        : {rows_scanned:,}")
print(f"Valid mentions counted     : {valid_mentions_counted:,}")
print(f"Missing mentions removed   : {missing_mentions_removed:,}")
print(f"Blank mentions removed     : {blank_mentions_removed:,}")
print(f"Total rows accounted for   : {accounted_entity_rows:,}")

print("\nAll scanned entity rows are fully accounted for.")

### Final Biomedical Mention Statistics

The final statistics distinguish between the complete number of extracted
entity rows and the valid normalised mention strings used in the frequency
analysis.

In [ ]:
# Creating Biomedical Mention Statistics


total_valid_mentions = int(
    entity_summary["frequency"].sum()
)

unique_mentions = int(
    len(entity_summary)
)

singleton_mentions = int(
    (entity_summary["frequency"] == 1).sum()
)

mean_valid_mentions_per_note = (
    total_valid_mentions
    / TOTAL_NOTES
)

if not entity_summary.empty:

    top_mention = str(
        entity_summary.iloc[0]["biomedical_mention"]
    )

    top_mention_frequency = int(
        entity_summary.iloc[0]["frequency"]
    )

else:

    top_mention = None
    top_mention_frequency = 0


entity_statistics = pd.DataFrame(
    {
        "statistic": [
            "Total entity rows scanned",
            "Valid normalised mentions",
            "Missing mention values removed",
            "Blank mentions removed",
            "Unique normalised mentions",
            "Singleton mentions",
            "Mean valid mentions per note",
            "Most frequent mention",
            "Most frequent mention count"
        ],
        "value": [
            rows_scanned,
            total_valid_mentions,
            missing_mentions_removed,
            blank_mentions_removed,
            unique_mentions,
            singleton_mentions,
            mean_valid_mentions_per_note,
            top_mention,
            top_mention_frequency
        ]
    }
)

print("-" * 75)
print("BIOMEDICAL MENTION STATISTICS")
print("-" * 75)

display(entity_statistics)

## 22. Export of Final scispaCy Summary Outputs

The final frequency table, biomedical mention statistics, and processing
statistics are exported for subsequent analysis and comparison with the
medSpaCy pipeline.

**Preparing the processing-statistics table**

In [ ]:
processing_statistics_df = pd.DataFrame(
    {
        "statistic": list(
            processing_statistics.keys()
        ),
        "value": list(
            processing_statistics.values()
        )
    }
)

print("-" * 75)
print("PROCESSING STATISTICS TABLE")
print("-" * 75)

display(processing_statistics_df)

**Defining output paths and export all three files**

In [ ]:
# Exporting Final Notebook 04 Summary Outputs

SUMMARY_OUTPUT_DIRECTORY = (
    ENTITY_SHARD_DIRECTORY.parent
    / "scispacy_summary_outputs"
)

SUMMARY_OUTPUT_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)


ENTITY_SUMMARY_FILE = (
    SUMMARY_OUTPUT_DIRECTORY
    / "entity_summary.csv.gz"
)

ENTITY_STATISTICS_FILE = (
    SUMMARY_OUTPUT_DIRECTORY
    / "entity_statistics.csv"
)

PROCESSING_STATISTICS_FILE = (
    SUMMARY_OUTPUT_DIRECTORY
    / "processing_statistics.csv"
)


entity_summary.to_csv(
    ENTITY_SUMMARY_FILE,
    index=False,
    compression="gzip"
)

entity_statistics.to_csv(
    ENTITY_STATISTICS_FILE,
    index=False
)

processing_statistics_df.to_csv(
    PROCESSING_STATISTICS_FILE,
    index=False
)


print("-" * 75)
print("FINAL OUTPUT EXPORT")
print("-" * 75)

print(f"Entity summary        : {ENTITY_SUMMARY_FILE}")
print(f"Entity statistics     : {ENTITY_STATISTICS_FILE}")
print(f"Processing statistics : {PROCESSING_STATISTICS_FILE}")

**Validating Final Outputs**

In [ ]:
required_output_files = [
    ENTITY_SUMMARY_FILE,
    ENTITY_STATISTICS_FILE,
    PROCESSING_STATISTICS_FILE
]

for output_file in required_output_files:

    assert output_file.exists(), (
        f"Missing output file: {output_file}"
    )

    assert output_file.stat().st_size > 0, (
        f"Output file is empty: {output_file}"
    )


# Reloading the corrected statistics file.
saved_entity_statistics = pd.read_csv(
    ENTITY_STATISTICS_FILE
)

missing_value_record = saved_entity_statistics.loc[
    saved_entity_statistics["statistic"]
    == "Missing mention values removed",
    "value"
]

assert len(missing_value_record) == 1, (
    "The corrected missing-value statistic was not found "
    "in entity_statistics.csv."
)

assert int(float(missing_value_record.iloc[0])) == 1622, (
    "The exported missing-value count is incorrect."
)


print("-" * 75)
print("FINAL VALIDATION")
print("-" * 75)

for output_file in required_output_files:

    file_size_kb = (
        output_file.stat().st_size
        / 1024
    )

    if file_size_kb >= 1024:

        print(
            f"{output_file.name:<32}"
            f"{file_size_kb / 1024:>10.2f} MB"
        )

    else:

        print(
            f"{output_file.name:<32}"
            f"{file_size_kb:>10.2f} KB"
        )

print("\nCorrected missing-value count: 1,622")
print("All outputs were created successfully.")